# Session 2 — Drift Alerting

**Goal:** Create a SQL alert that automatically notifies you when drift exceeds a threshold.

The alert setup:
1. Write a SQL query that returns rows when PSI > threshold
2. Create a Databricks SQL alert on that query
3. Subscribe via email (or Slack webhook)
4. Alert fires automatically after each monitoring refresh

> **Before running the alert query at the bottom of this notebook**, make sure the monitoring
> refresh from `03_monitoring_setup.ipynb` has completed (2-3 minutes). The `drift_metrics`
> table must exist and have data for the alert to trigger correctly.

In [0]:
%run ../utils/config


## The Alert Query

This query returns results (triggering the alert) when any feature has PSI > 0.2.

In [0]:
alert_sql = f"""
SELECT
  column_name,
  drift_type,
  ROUND(chi_squared_test.statistic, 4) AS psi_score,
  window.start AS window_start
FROM `{catalog}`.`{schema}`.churn_inference_log_drift_metrics
WHERE chi_squared_test.statistic > 0.2
  AND drift_type = 'CONSECUTIVE'
ORDER BY psi_score DESC
"""
print("Alert query:")
print(alert_sql)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import CreateQueryRequestQuery
w = WorkspaceClient()

# Try to find a warehouse automatically
warehouses = list(w.warehouses.list())
if warehouses:
    warehouse_id = warehouses[0].id
    print(f"Using warehouse: {warehouses[0].name} ({warehouse_id})")
else:
    raise ValueError("No SQL warehouse found. Ask the instructor for the warehouse ID.")

# Create the alert query
query = w.queries.create(
    query=CreateQueryRequestQuery(
        query_text=alert_sql,
        display_name=f"churn_drift_alert_{schema}",
        warehouse_id=warehouse_id,
        description=f"Fires when any feature PSI > 0.2 for {schema}",
    )
)
print(f"✓ Query created: {query.id}")
print(f"  Name: {query.display_name}")

In [0]:
# Create an alert on the query
# Alert fires when the query returns any rows (drift detected)
from databricks.sdk.service.sql import (
    CreateAlertRequestAlert,
    AlertCondition,
    AlertConditionOperand,
    AlertConditionThreshold,
    AlertOperandColumn,
    AlertOperandValue,
    AlertOperator,
)

alert = w.alerts.create(
    alert=CreateAlertRequestAlert(
        display_name=f"Churn Drift Alert — {schema}",
        condition=AlertCondition(
            op=AlertOperator.GREATER_THAN,
            operand=AlertConditionOperand(
                column=AlertOperandColumn(name="psi_score")
            ),
            threshold=AlertConditionThreshold(
                value=AlertOperandValue(double_value=0.2)
            ),
        ),
        query_id=query.id,
        seconds_to_retrigger=3600,  # Re-alert after 1 hour if still drifting
    )
)
print(f"✓ Alert created: {alert.id}")
print(f"  Name: {alert.display_name}")

## Subscribe to the Alert

1. Go to **SQL → Alerts** in the left sidebar
2. Find your `Churn Drift Alert — <schema>`
3. Click **+ Add notification destination**
4. Select **Email** and enter your email address
5. Click **Save**

The alert will fire after the next monitoring refresh detects PSI > 0.2.

## Discussion

- **At what PSI threshold do you retrain vs just alert?** (0.1 = warning, 0.25 = critical?)
- **How do you distinguish data pipeline issues from real drift?**
  (Check ETL logs first — is the null rate suddenly 0%?)
- **Who owns this alert runbook at 2am?**
  (Define on-call rotation during client handover)

➡️ Next: `05_trigger_retrain.ipynb` — trigger the retrain pipeline and review the deployment gate